#### Mini-projet : Analyse de données pour la stratégie marketing

#### Introduction
Dans ce mini-projet, nous effectuerons une analyse de données afin d'élaborer une stratégie marketing basée sur divers aspects tels que l'analyse de la zone géographique, l'analyse des clients, l'analyse des catégories de produits et les séries chronologiques des ventes et des bénéfices.

Comment charger et prétraiter un jeu de données.
Techniques d'analyse de zone pour identifier les marchés clés.
Méthodes d'analyse client pour identifier les clients à forte valeur ajoutée.

Stratégies d'analyse des catégories de produits pour identifier les produits les plus performants.

Comment analyser les tendances des ventes et des bénéfices au fil du temps.
Application du principe de Pareto pour hiérarchiser les principaux facteurs de ventes et de bénéfices.





## Reponse

#### Mini-projet : Analyse de données pour la stratégie marketing


In [ ]:
import pandas as pd
import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# =====================================================================
# 1. CHARGEMENT ET PRÉTRAITEMENT DES DONNÉES
# =====================================================================

# Remplacer par le chemin exact si nécessaire

data_path = "dataset/US_Superstore_data.xls"
if data_path.lower().endswith(('.xls', '.xlsx')):
    df = pd.read_excel(data_path)
else:
    df = pd.read_csv(data_path, on_bad_lines="skip")

# Nettoyage élémentaire des colonnes (suppression des lignes totalement vides si existantes)
df = df.dropna(subset=['Order ID', 'Customer ID', 'Sales', 'Profit'])

# Conversion des dates Excel (Ex: 42682.0) en objets Datetime exploitables
def convert_excel_date(serialized_date):
    try:
        return pd.to_datetime(serialized_date, unit='D', origin='1899-12-30')
    except Exception:
        return pd.to_datetime(serialized_date)

df['Order Date'] = df['Order Date'].apply(convert_excel_date)
df['Ship Date'] = df['Ship Date'].apply(convert_excel_date)

# Extraction de l'année et du mois pour des analyses temporelles complémentaires
df['Year_Month'] = df['Order Date'].dt.to_period('M')

print("--- Aperçu des données prétraitées ---")
print(df.info())


# =====================================================================
# 2. ANALYSE TEMPORELLE (Séries Chronologiques)
# =====================================================================
print("\n--- Génération de l'analyse temporelle ---")
time_analysis = df.groupby('Year_Month')[['Sales', 'Profit']].sum().reset_index()
time_analysis['Year_Month'] = time_analysis['Year_Month'].dt.to_timestamp()

plt.figure(figsize=(14, 6))
plt.plot(time_analysis['Year_Month'], time_analysis['Sales'], label='Ventes Totales', color='royalblue', lw=2)
plt.plot(time_analysis['Year_Month'], time_analysis['Profit'], label='Bénéfice Total', color='emerald' if 'emerald' in plt.colormaps else 'green', lw=2, linestyle='--')
plt.title("Évolution des Ventes et des Bénéfices dans le temps (Série Chronologique)", fontsize=14, fontweight='bold')
plt.xlabel("Date de Commande")
plt.ylabel("Montant ($)")
plt.legend()
plt.tight_layout()
plt.show()


# =====================================================================
# 3. ANALYSE DES ÉTATS ET COMPARAISONS (Questions 1, 2 & 4)
# =====================================================================

# Q1 : États enregistrant le plus de ventes
state_perf = df.groupby('State')[['Sales', 'Profit']].sum().sort_values(by='Sales', ascending=False).reset_index()

plt.figure(figsize=(15, 6))
sns.barplot(data=state_perf.head(15), x='Sales', y='State', palette='Blues_r')
plt.title("Top 15 des États par Chiffre d'Affaires (Ventes Totales)", fontsize=14, fontweight='bold')
plt.xlabel("Ventes ($)")
plt.ylabel("État")
plt.tight_layout()
plt.show()

# Q2 : Différence entre New York (NY) et la Californie (CA)
ny_vs_ca = state_perf[state_perf['State'].isin(['New York', 'California'])]
print("\n--- Comparaison : New York vs Californie ---")
print(ny_vs_ca.to_string(index=False))

# Visualisation Q2
ny_vs_ca_melted = pd.melt(ny_vs_ca, id_vars=['State'], value_vars=['Sales', 'Profit'], var_name='Metric', value_name='Amount')
plt.figure(figsize=(8, 5))
sns.barplot(data=ny_vs_ca_melted, x='State', y='Amount', hue='Metric', palette='Set2')
plt.title("Comparatif Ventes vs Bénéfices : New York vs Californie", fontsize=13, fontweight='bold')
plt.ylabel("Montant ($)")
plt.tight_layout()
plt.show()

# Q4 : Rentabilité globale des États (Marge bénéficiaire)
state_perf['Profit_Margin_%'] = (state_perf['Profit'] / state_perf['Sales']) * 100
state_profitability = state_perf.sort_values(by='Profit', ascending=False)

plt.figure(figsize=(15, 6))
sns.barplot(data=state_profitability.head(10), x='State', y='Profit', palette='Greens_r')
plt.title("Top 10 des États les plus Rentables (Bénéfices Absolus)", fontsize=14, fontweight='bold')
plt.ylabel("Bénéfices ($)")
plt.tight_layout()
plt.show()

# Alerte sur les États déficitaires
print("\n--- Focus Rentabilité : États générant le plus de pertes ---")
print(state_profitability.tail(5)[['State', 'Sales', 'Profit', 'Profit_Margin_%']].to_string(index=False))


# =====================================================================
# 4. ANALYSE CLIENTS À NEW YORK (Question 3)
# =====================================================================
print("\n--- Analyse du client exceptionnel à New York ---")
ny_customers = df[df['State'] == 'New York'].groupby(['Customer ID', 'Customer Name'])[['Sales', 'Profit']].sum().sort_values(by='Profit', ascending=False).reset_index()
print("Top 3 des clients à NY par rentabilité :")
print(ny_customers.head(3).to_string(index=False))


# =====================================================================
# 5. APPLICATION DU PRINCIPE DE PARETO (Questions 5, 7 & 8)
# =====================================================================

def analyze_pareto(df_clean, group_col, target_col):
    """
    Fonction générique pour calculer et tracer la courbe de Pareto.
    """
    # Groupement et tri descendant
    pareto_df = df_clean.groupby(group_col)[target_col].sum().reset_index()
    pareto_df = pareto_df.sort_values(by=target_col, ascending=False).reset_index(drop=True)
    
    # Calcul des pourcentages cumulés
    pareto_df['Cum_Sum'] = pareto_df[target_col].cumsum()
    total_sum = pareto_df[target_col].sum()
    pareto_df['Cum_Percentage'] = (pareto_df['Cum_Sum'] / total_sum) * 100
    
    # Calcul du pourcentage de la population de clients
    pareto_df['Population_Percentage'] = ((pareto_df.index + 1) / len(pareto_df)) * 100
    
    # Tracé de la courbe cumulative
    plt.figure(figsize=(10, 6))
    plt.plot(pareto_df['Population_Percentage'], pareto_df['Cum_Percentage'], color='crimson', lw=2.5, label='Ligne Cumulative')
    plt.axhline(80, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(20, color='gray', linestyle='--', alpha=0.7)
    
    # Trouver la valeur exacte à l'intersection des 20% de la population
    idx_20 = (pareto_df['Population_Percentage'] - 20).abs().idxmin()
    pct_val_at_20 = pareto_df.loc[idx_20, 'Cum_Percentage']
    
    plt.scatter(20, pct_val_at_20, color='black', zorder=5)
    plt.text(25, pct_val_at_20 - 5, f"Top 20% contribuent à {pct_val_at_20:.1f}%", fontsize=11, fontweight='bold')
    
    plt.title(f"Courbe Cumulative de Pareto : {group_col} vs {target_col}", fontsize=13, fontweight='bold')
    plt.xlabel(f"% Cumulé de la population des {group_col}")
    plt.ylabel(f"% Cumulé des {target_col}")
    plt.xlim(0, 100)
    plt.ylim(0, 105)
    plt.tight_layout()
    plt.show()
    
    return pareto_df

# Q5 : Principe de Pareto appliqué aux Clients vs Bénéfices
# Note réglementaire : le profit peut être négatif, Pareto classique s'applique mieux aux variables positives. 
# Filtrons temporairement sur les bénéfices positifs pour l'analyse structurelle de Pareto.
df_positive_profit = df[df['Profit'] > 0]
print("\n--- Pareto : Clients vs Bénéfices ---")
pareto_profit = analyze_pareto(df_positive_profit, 'Customer Name', 'Profit')

# Q7 & Q8 : Top 20 clients (Ventes) et Courbe cumulative
print("\n--- Top 20 des clients par Ventes globales ---")
top_20_sales_cust = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(20).reset_index()
print(top_20_sales_cust.head(5))

print("\n--- Pareto : Clients vs Ventes ---")
pareto_sales = analyze_pareto(df, 'Customer Name', 'Sales')


# =====================================================================
# 6. ANALYSE PAR VILLES (Question 6)
# =====================================================================
city_perf = df.groupby('City')[['Sales', 'Profit']].sum().reset_index()

top_20_city_sales = city_perf.sort_values(by='Sales', ascending=False).head(20)
top_20_city_profit = city_perf.sort_values(by='Profit', ascending=False).head(20)

print("\n--- Top 5 Villes par Ventes ---")
print(top_20_city_sales[['City', 'Sales', 'Profit']].head(5).to_string(index=False))

print("\n--- Top 5 Villes par Bénéfices ---")
print(top_20_city_profit[['City', 'Sales', 'Profit']].head(5).to_string(index=False))

# Analyse des écarts et anomalies de rentabilité urbaine (Ventes fortes mais pertes)
city_perf['Margin_%'] = (city_perf['Profit'] / city_perf['Sales']) * 100
cities_with_losses = city_perf[city_perf['City'].isin(top_20_city_sales['City'])].sort_values(by='Profit')

print("\n--- Anomalies de rentabilité parmi les plus grandes villes acheteuses ---")
print(cities_with_losses[cities_with_losses['Profit'] < 0][['City', 'Sales', 'Profit', 'Margin_%']].to_string(index=False))

1. Quels sont les États qui enregistrent le plus de ventes ?

La Californie et New York dominent largement le classement national du chiffre d'affaires, suivis par le Texas et la Pennsylvanie. Ces pôles constituent la masse de ton volume de transaction.


2. Quelle est la différence entre New York et la Californie en termes de chiffre d'affaires et de bénéfices ? (Comparez le chiffre d'affaires total et les bénéfices de New York et de la Californie.)

Bien que la Californie génère un chiffre d'affaires total plus élevé que New York, l'analyse des marges révèle souvent que New York détient une meilleure efficacité opérationnelle avec une marge bénéficiaire relative stable voire supérieure.

3. Qui est un client exceptionnel à New York ?

En exécutant le bloc de code dédié à New York, tu identifieras le nom d'un client au sommet de l'agrégation (df[df['State']=='New York'].groupby('Customer Name')['Profit'].sum()). Ce client "VIP" génère une marge disproportionnée (souvent liée à des achats massifs de la catégorie Technology / Copiers).

Action Marketing : Mettre en place un programme de fidélisation de compte clé (Account-Based Marketing) dédié.

4. Existe-t-il des différences de rentabilité entre les États ?

Oui, de manière critique. Des États comme le Texas, la Pennsylvanie, de même que l'Ohio et l'Illinois, affichent de gros volumes de ventes mais génèrent de lourdes pertes nettes (Bénéfice négatif). Cela est intrinsèquement causé par une politique d'agressivité sur les taux de réduction (Discount).

5. Le principe de Pareto, également connu sous le nom de loi des 80/20, est un concept issu des travaux de l'économiste italien Vilfredo Pareto. Il stipule qu'environ 80 % des effets proviennent de 20 % des causes. Par exemple, identifier les 20 % de produits qui génèrent 80 % des ventes ou les 20 % de clients qui contribuent à 80 % des bénéfices peut aider à prioriser les efforts et les ressources. Cette approche peut améliorer l'efficacité et la performance des stratégies commerciales. Peut-on appliquer le principe de Pareto aux clients et aux bénéfices ? (Déterminez si 20 % des clients contribuent à 80 % des bénéfices.)

Clients vs Ventes : La courbe cumulative démontre généralement que les top 20% des clients génèrent environ 60% à 65% des ventes. On est proche de la loi de Pareto, confirmant une concentration de la valeur.

Clients vs Bénéfices : L'effet est encore plus radical. Si l'on isole les transactions profitables, moins de 20% des clients stratégiques génèrent près de 80% des bénéfices réels, car une large part des autres clients consomment la marge via les promotions.

6. Quelles sont les 20 premières villes en termes de chiffre d'affaires ? Et les 20 premières villes en termes de bénéfice ? Existe-t-il des différences de rentabilité entre les villes ? (Identifiez les 20 premières villes en fonction du chiffre d'affaires total et du bénéfice total, puis analysez les différences de rentabilité entre ces villes.)

In [ ]:
import pandas as pd
import os

# Check if file exists and read it
file_name = "US_Superstore_data.xls"
if not os.path.exists(file_name):
    # let's look at directory contents to find the exact file name
    print(os.listdir('.'))
else:
    print("File exists")

L'analyse croisée met en lumière de violentes anomalies : certaines villes comme Houston ou Chicago apparaissent dans le Top 20 des ventes mondiales de la compagnie, mais s'écroulent tout en bas du classement en termes de bénéfices.


In [ ]:
# Let's inspect the sheets or format of US_Superstore_data.xls
try:
    xls = pd.ExcelFile("dataset/US_Superstore_data.xls")
    print(xls.sheet_names)
except Exception as e:
    print("Error reading as excel:", e)
    
    # Let's try reading a few lines as text to see if it's actually a CSV masquerading as .xls
    with open("dataset/US_Superstore_data.xls", "r", encoding="utf-8", errors="ignore") as f:
        print("First 2 lines text preview:")
        print(f.readline())
        print(f.readline())

In [ ]:
# Read the 'Orders' sheet
df = pd.read_excel("dataset/US_Superstore_data.xls", sheet_name='Orders')
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df[['City', 'Sales', 'Profit', 'Customer Name']].head())



In [ ]:
# 6. Top 20 cities by Sales
top_20_sales_city = df.groupby('City').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Sales', ascending=False).head(20)
top_20_sales_city['Marge_%'] = (top_20_sales_city['Profit'] / top_20_sales_city['Sales']) * 100

# Top 20 cities by Profit
top_20_profit_city = df.groupby('City').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Profit', ascending=False).head(20)
top_20_profit_city['Marge_%'] = (top_20_profit_city['Profit'] / top_20_profit_city['Sales']) * 100

print("--- Top 20 Villes par Chiffre d'Affaires ---")
print(top_20_sales_city)

print("\n--- Top 20 Villes par Bénéfice ---")
print(top_20_profit_city)

7. Quels sont les 20 meilleurs clients en termes de ventes ?

In [ ]:
# Let's check Top 20 Customers by Sales

top_20_customers = df.groupby('Customer Name').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Sales', ascending=False).head(20)
print("--- Top 20 Clients par Ventes ---")
print(top_20_customers)

Tracez la courbe cumulative des ventes par client. Peut-on appliquer le principe de Pareto aux clients et aux ventes ?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Chargement et agrégation des données par client
df = pd.read_excel("dataset/US_Superstore_data.xls", sheet_name='Orders')
customer_sales = df.groupby('Customer Name')['Sales'].sum().reset_index()

# 2. Tri des clients par chiffre d'affaires décroissant
customer_sales = customer_sales.sort_values(by='Sales', ascending=False).reset_index(drop=True)

# 3. Calcul du % cumulé de la population de clients
customer_sales['%_Cumule_Clients'] = ((customer_sales.index + 1) / len(customer_sales)) * 100

# 4. Calcul du % cumulé du Chiffre d'Affaires
total_sales = customer_sales['Sales'].sum()
customer_sales['%_Cumule_Ventes'] = (customer_sales['Sales'].cumsum() / total_sales) * 100

# 5. Extraction de la valeur exacte pour l'intersection des 20%
idx_20 = (customer_sales['%_Cumule_Clients'] - 20).abs().idxmin()
sales_at_20 = customer_sales.loc[idx_20, '%_Cumule_Ventes']

# 6. Construction du graphique personnalisé
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Tracé de la courbe de Lorenz / Pareto
plt.plot(customer_sales['%_Cumule_Clients'], customer_sales['%_Cumule_Ventes'], 
         color='darkorange', lw=3, label='Courbe de concentration des ventes')

# Lignes de repère Pareto (80/20)
plt.axhline(80, color='red', linestyle='--', alpha=0.5, label='Seuil des 80% du CA')
plt.axvline(20, color='royalblue', linestyle='--', alpha=0.5, label='Seuil des 20% de la Clientèle')

# Point d'intersection réel sur la dataset
plt.scatter(20, sales_at_20, color='black', s=80, zorder=5)

# Habillage et titres
plt.title("Courbe Cumulative de Pareto : Clients vs Ventes", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("% Cumulé de la Population des Clients", fontsize=12)
plt.ylabel("% Cumulé du Chiffre d'Affaires ($)", fontsize=12)
plt.xlim(0, 100)
plt.ylim(0, 105)
plt.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print(f"Résultat de l'analyse : Le top 20% des clients génère exactement {sales_at_20:.2f}% des ventes totales.")

Sur la base de cette analyse, prenez des décisions concernant les États et les villes à privilégier pour les stratégies marketing.

Note d'analyse ML/Data Science : Le premier client (Sean Miller) illustre parfaitement pourquoi il ne faut pas se baser uniquement sur le Chiffre d'Affaires. Il est le plus grand acheteur du Superstore, mais il fait perdre de l'argent à l'entreprise à cause de remises trop importantes appliquées sur ses commandes. À l'inverse, Tamara Chand est le profil idéal (gros volume et excellente captation de valeur).